In [ ]:
# Parameters
city_key = "ywg"
analysis_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

# PR2: Route Tier Classification

This notebook trains an interpretable classifier that predicts PTN tier from route-level schedule features. The goal is validation and explanation, not a production model.

## 1. Setup & Helpers

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from ptn_analysis import TransitContext

## 2. Load Route Features

In [ ]:
ctx = TransitContext.from_defaults(feed_id=analysis_feed_id)
frequency_analyzer = ctx.frequency()
route_feature_table = frequency_analyzer.build_route_classification_feature_table()
print(f"Route feature rows: {len(route_feature_table)}")
display(route_feature_table.head())

## 3. Fit Classifier

In [ ]:
# Route tier classifier — compute results for combined figure
if not route_feature_table.empty:
    from shapely.geometry import LineString
    import contextily as cx
    import geopandas as gpd
    from sklearn.impute import SimpleImputer

    feature_columns = [
        "scheduled_trip_count", "mean_headway_minutes", "scheduled_speed_kmh",
        "mean_trip_distance", "mean_trip_duration", "passup_count",
        "avg_deviation_seconds", "peak_hour_departures", "daily_departures",
    ]
    imputer = SimpleImputer(strategy="median")
    feature_array = imputer.fit_transform(route_feature_table[feature_columns])
    feature_table = pd.DataFrame(feature_array, columns=feature_columns)

    label_encoder = LabelEncoder()
    target_values = label_encoder.fit_transform(route_feature_table["ptn_tier"].fillna("Unknown"))
    class_counts = pd.Series(target_values).value_counts()
    stratify_values = target_values if len(class_counts) > 1 and class_counts.min() >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        feature_table, target_values, test_size=0.3, random_state=42, stratify=stratify_values,
    )
    classifier = RandomForestClassifier(random_state=42, n_estimators=300)
    classifier.fit(x_train, y_train)
    predicted_values = classifier.predict(x_test)
    unique_labels = sorted(set(y_test) | set(predicted_values))
    display_labels = label_encoder.inverse_transform(unique_labels)
    weighted_f1 = f1_score(y_test, predicted_values, average="weighted")
    print(f"Route Tier RF — Weighted F1: {weighted_f1:.3f}")

    tier_importance = pd.DataFrame({
        "feature": feature_columns, "importance": classifier.feature_importances_,
    }).sort_values("importance", ascending=True)
    tier_confusion = confusion_matrix(y_test, predicted_values, labels=unique_labels)

    # Route GDF for spatial panel
    shape_routes = ctx.working_db.query("""
        SELECT r.route_short_name, t.shape_id, COUNT(*) as trip_count
        FROM ywg_trips t JOIN ywg_routes r ON t.route_id = r.route_id AND t.feed_id = r.feed_id
        WHERE t.feed_id = 'current' AND t.shape_id IS NOT NULL
        GROUP BY r.route_short_name, t.shape_id ORDER BY r.route_short_name, trip_count DESC
    """)
    best_shapes = shape_routes.drop_duplicates(subset="route_short_name", keep="first")
    shape_ids = best_shapes["shape_id"].unique().tolist()
    shapes_df = ctx.working_db.query(f"""
        SELECT shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence FROM ywg_shapes
        WHERE shape_id IN ({','.join(f"'{s}'" for s in shape_ids)}) ORDER BY shape_id, shape_pt_sequence
    """)
    lines = {}
    for sid, g in shapes_df.groupby("shape_id"):
        coords = list(zip(g["shape_pt_lon"], g["shape_pt_lat"]))
        if len(coords) >= 2: lines[sid] = LineString(coords)
    best_shapes = best_shapes.copy()
    best_shapes["geometry"] = best_shapes["shape_id"].map(lines)
    route_gdf = gpd.GeoDataFrame(best_shapes.dropna(subset=["geometry"]), geometry="geometry", crs="EPSG:4326")
    route_feature_table = route_feature_table.copy()
    route_feature_table["predicted_ptn_tier"] = label_encoder.inverse_transform(classifier.predict(feature_table))
    from ptn_analysis.analysis import PTN_TIER_COLORS
    tier_map_df = route_gdf.merge(
        route_feature_table[["route_short_name", "ptn_tier", "predicted_ptn_tier"]],
        on="route_short_name", how="inner",
    ).to_crs(epsg=3857)
    tier_map_df["correct"] = tier_map_df["ptn_tier"] == tier_map_df["predicted_ptn_tier"]
    tier_neigh_gdf = ctx.working_db.neighbourhood_gdf().to_crs(epsg=3857)
    print(f"Route tier data ready: {len(tier_map_df)} routes")
else:
    tier_importance = None


## 3. Interpretation: Service Typology Consistency Audit

**Note on classifier interpretation:** PTN tiers are administratively defined by route naming conventions in `views.sql` (e.g., `BLUE` → Rapid Transit, `FX*` → Frequent Express, `F*` → Frequent, 3-digit → Community). High classifier accuracy reflects that the PTN naming scheme is **internally consistent** with service parameters — not that routes have truly distinct behavioural patterns discovered by the model.

This classifier is best interpreted as a **consistency audit**: routes the model misclassifies are operating inconsistently with their assigned tier's service parameters (e.g., a "Frequent" route running at 30-min headways).

High weighted F1 (>0.85) confirms the PTN tier assignments are administratively coherent. Misclassified routes are candidates for service review.

## 4. Neighbourhood Coverage Classification (Equity Question)

Can census demographics predict neighbourhood transit coverage quality? This answers PR2 Research Question 4.

**Feature contract (no leakage):**
- Target: `density_category` (High/Medium/Low) from neighbourhood transit access metrics
- Predictors: census-only demographics (exogenous to the transit system)
- Excluded: `stop_count`, `stop_density_per_km2`, `priority_rank` (these define or derive from the target)

In [ ]:
# --- classification_combined.png: Clean 2x2 layout ---
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import balanced_accuracy_score, classification_report as clf_report
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix as cm_fn
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.impute import SimpleImputer
import numpy as np

# --- Coverage classifier data ---
ctx_cov = TransitContext.from_defaults(feed_id=analysis_feed_id)
cov_db = ctx_cov.working_db
access_tn = cov_db.table_name("neighbourhood_transit_access_metrics", city_key)
census_tn = cov_db.table_name("census_by_neighbourhood", city_key)
access_t = cov_db.query(f"SELECT neighbourhood_id, neighbourhood, density_category FROM {access_tn} WHERE feed_id = :f", {"f": analysis_feed_id}) if cov_db.relation_exists(access_tn) else pd.DataFrame()
census_t = cov_db.query(f"SELECT neighbourhood_id, population_density_per_km2, pct_commute_public_transit, median_household_income_2020, pct_seniors_65_plus FROM {census_tn}") if cov_db.relation_exists(census_tn) else pd.DataFrame()
nb_ft = access_t.merge(census_t, on="neighbourhood_id", how="inner") if not access_t.empty and not census_t.empty else pd.DataFrame()

has_tier = tier_importance is not None
has_cov = not nb_ft.empty and "density_category" in nb_ft.columns

if not has_tier and not has_cov:
    save_placeholder_figure("classification_combined.png", "Classification data not available.",
                            figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # === TOP ROW: Route tier classifier ===
    if has_tier:
        # Compute balanced accuracy from confusion matrix
        tier_per_class = tier_confusion.diagonal() / tier_confusion.sum(axis=1).clip(min=1)
        tier_bal_acc = tier_per_class.mean()

        axes[0, 0].barh(tier_importance["feature"], tier_importance["importance"], color="#7570b3")
        axes[0, 0].set_title("Feature Importance (Route Tier)", fontsize=11, fontweight="bold")
        axes[0, 0].set_xlabel("Importance")

        ConfusionMatrixDisplay(confusion_matrix=tier_confusion, display_labels=display_labels).plot(
            ax=axes[0, 1], xticks_rotation=45, colorbar=False)
        axes[0, 1].set_title(f"Confusion Matrix (Route Tier)\nBalanced Acc = {tier_bal_acc:.2f}",
                             fontsize=11, fontweight="bold")
    else:
        for j in range(2):
            axes[0, j].axis("off")

    # === BOTTOM ROW: Coverage classifier ===
    if has_cov:
        cov_features = ["population_density_per_km2", "pct_commute_public_transit",
                        "median_household_income_2020", "pct_seniors_65_plus"]
        cov_feature_labels = ["Pop Density/km\u00b2", "% Transit Commute",
                              "Median Income 2020", "% Seniors 65+"]
        cov_imp = SimpleImputer(strategy="median")
        X_cov = cov_imp.fit_transform(nb_ft[cov_features])
        y_cov_raw = nb_ft["density_category"].fillna("Low")
        cov_enc = LabelEncoder()
        y_cov = cov_enc.fit_transform(y_cov_raw)
        min_cls = pd.Series(y_cov).value_counts().min()
        n_sp = min(5, int(min_cls))
        cov_clf = RFC(random_state=42, n_estimators=300, class_weight="balanced")
        if n_sp >= 2:
            cv = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=42)
            y_pred_cov = cross_val_predict(cov_clf, X_cov, y_cov, cv=cv)
            cov_clf.fit(X_cov, y_cov)
        else:
            cov_clf.fit(X_cov, y_cov)
            y_pred_cov = cov_clf.predict(X_cov)
        bal_acc_cov = balanced_accuracy_score(y_cov, y_pred_cov)

        cov_imp_df = pd.DataFrame({
            "feature": cov_feature_labels,
            "importance": cov_clf.feature_importances_,
        }).sort_values("importance", ascending=True)
        axes[1, 0].barh(cov_imp_df["feature"], cov_imp_df["importance"], color="#d95f02")
        axes[1, 0].set_title("Feature Importance (Coverage)", fontsize=11, fontweight="bold")
        axes[1, 0].set_xlabel("Importance")

        ConfusionMatrixDisplay(
            confusion_matrix=cm_fn(y_cov, y_pred_cov), display_labels=cov_enc.classes_,
        ).plot(ax=axes[1, 1], xticks_rotation=30, colorbar=False)
        axes[1, 1].set_title(f"Confusion Matrix (Coverage)\nBalanced Acc = {bal_acc_cov:.2f}",
                             fontsize=11, fontweight="bold")
    else:
        for j in range(2):
            axes[1, j].axis("off")

    plt.tight_layout()
    save_report_figure(fig, "classification_combined.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)
